# Monte Carlo Fitted Policy Iteration

This notebook runs an end-to-end evaluate → fit → greedify loop for the **controlled fantasy team** while every opponent remains on `RosterAwareSoftmaxPolicy`.

Production defaults perform three rounds with 20,000 fresh post-pick states per round:

1. Evaluate the controlled `RosterAwareSoftmaxPolicy`, then fit $\hat V^{\pi_0}$.
2. Improve to $\pi_1(s)=\arg\max_a \hat V^{\pi_0}(T(s,a))$, collect fresh $\pi_1$ trajectories, and fit $\hat V^{\pi_1}$.
3. Repeat once more to obtain $\pi_2$.

This is **Monte Carlo fitted/generalized policy iteration**, not exact tabular policy iteration. Values are estimated with finite simulations and function approximation, so improvement is measured on a fixed held-out paired benchmark rather than assumed.

## 1. Setup

Install the ML extras once with `pip install -e ".[ml]"`. In Colab, clone or upload the repository first, then run this notebook from the repository so `pyproject.toml` is discoverable.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        "Run this notebook from the fantasy-drafter repository (pyproject.toml not found)."
    )
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from fantasy_draft.policy_iteration import (
    GreedyValuePolicy,
    PolicyIterationConfig,
    run_fitted_policy_iteration,
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Repository: {ROOT}")
print(f"Training/inference device: {DEVICE}")

## 2. Configure the loop

`SMOKE_TEST=False` is the requested production run: 60,000 total rows across three fitted-policy rounds. Completed rounds are resumable, so restarting the cell will not regenerate valid artifacts.

Each row is a controlled-team post-pick state from a complete trajectory. All states from that trajectory receive the same Monte Carlo estimate of its terminal roster's expected weekly score. Draft-grouped splitting prevents states from the same player pool/history leaking across train, validation, and test sets.

In [ ]:
SMOKE_TEST = False

if SMOKE_TEST:
    # End-to-end correctness check that finishes quickly.
    config = PolicyIterationConfig(
        iterations=2,
        samples_per_iteration=240,
        season_simulations=3,
        epsilon=0.10,
        benchmark_drafts=6,
        benchmark_seasons=3,
        seed=42,
        max_epochs=5,
        patience=2,
        batch_size=128,
        data_dir=ROOT / "data" / "policy_iteration_smoke",
        model_dir=ROOT / "models" / "policy_iteration_smoke",
        resume=True,
    )
else:
    # Production fitted-policy-iteration run requested for this project.
    config = PolicyIterationConfig(
        iterations=3,
        samples_per_iteration=20_000,
        season_simulations=25,
        epsilon=0.10,
        benchmark_drafts=96,
        benchmark_seasons=100,
        seed=42,
        max_epochs=100,
        patience=12,
        batch_size=512,
        learning_rate=1e-3,
        weight_decay=1e-4,
        data_dir=ROOT / "data" / "policy_iteration",
        model_dir=ROOT / "models" / "policy_iteration",
        resume=True,
    )

print(config)

## 3. Run evaluation, training, and greedy improvement

Iteration 0 evaluates the roster-aware baseline. Every later iteration loads the preceding checkpoint and uses its greedy post-action policy for the controlled team. Opponents remain roster-aware throughout, matching the best-response objective selected for this experiment.

Fresh labels are generated each round; older policy labels are not mixed into the new value target.

In [ ]:
iteration_summary = run_fitted_policy_iteration(
    config,
    device=DEVICE,
    verbose=True,
)

display(iteration_summary.round(4))
print(f"\nRecommended checkpoint: {config.model_dir / 'best_value.pt'}")

## 4. Diagnose value fitting and policy improvement

Low prediction error is not enough: the greedy policy can exploit approximation errors. The held-out paired benchmark is therefore the primary policy-selection signal. Every iteration uses the same benchmark seeds and compares its controlled policy against `RosterAwareSoftmaxPolicy` in the same slot and player pool.

In [ ]:
metrics = []
for iteration in range(config.iterations):
    metrics_path = config.data_dir / f"metrics_{iteration:02d}.json"
    with open(metrics_path, encoding="utf-8") as handle:
        metrics.append(json.load(handle))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
iterations = [item["iteration"] for item in metrics]

for item in metrics:
    history = pd.DataFrame(item["history"])
    axes[0].plot(history["epoch"], history["val_loss"], label=f"iter {item['iteration']}")
axes[0].set_title("Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Standardized SmoothL1")
axes[0].legend()

advantages = [item["benchmark"]["paired_advantage"] for item in metrics]
ci_low = [item["benchmark"]["paired_advantage_ci_low"] for item in metrics]
ci_high = [item["benchmark"]["paired_advantage_ci_high"] for item in metrics]
axes[1].errorbar(
    iterations,
    advantages,
    yerr=[np.array(advantages) - ci_low, np.array(ci_high) - advantages],
    marker="o",
    capsize=4,
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Paired advantage vs baseline")
axes[1].set_xlabel("Fitted-policy iteration")
axes[1].set_ylabel("Expected points/week")

candidate_values = [item["benchmark"]["candidate_value"] for item in metrics]
baseline_values = [item["benchmark"]["baseline_value"] for item in metrics]
axes[2].plot(iterations, candidate_values, marker="o", label="greedy MLP")
axes[2].plot(iterations, baseline_values, marker="o", label="roster-aware baseline")
axes[2].set_title("Held-out policy value")
axes[2].set_xlabel("Fitted-policy iteration")
axes[2].set_ylabel("Expected points/week")
axes[2].legend()

for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. Load the recommended policy

The recommended checkpoint is selected by mean held-out candidate value, not by neural-network test error. It can be used anywhere a draft policy is accepted. The `rng` argument is ignored because greedy checkpoint inference is deterministic.

In [ ]:
with open(config.data_dir / "manifest.json", encoding="utf-8") as handle:
    manifest = json.load(handle)

best_checkpoint = Path(manifest["recommended_checkpoint"])
best_policy = GreedyValuePolicy(best_checkpoint, device=DEVICE, name="recommended_fitted_policy")

print(f"Recommended iteration: {manifest['recommended_iteration']}")
print(f"Checkpoint: {best_checkpoint}")
print(f"Policy name: {best_policy.name}")

## Interpretation

This loop has the policy-evaluation and greedy-improvement structure of policy iteration, but it is not guaranteed to improve monotonically:

- Monte Carlo labels contain finite-sample noise.
- The MLP introduces approximation error.
- Greedy maximization can exploit value overestimation on unusual actions.
- Epsilon exploration changes the evaluated behavior policy slightly.
- Opponents are held fixed, so the result is a best response—not a self-play equilibrium.

Use the held-out paired benchmark and confidence interval to decide whether a later checkpoint is actually better. If performance plateaus or regresses, increase trajectory diversity and season simulations before increasing network size.